In [1]:
from pathlib import Path
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, theilslopes

FILE = "Water Quality.csv"  # Set to full path if needed
OUT = "task1_mann_kendall_outputs"
ALPHA = 0.05
MIN_MONTHS = 8

def mann_kendall(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 3:
        return n, np.nan, np.nan, np.nan, "insufficient data"
    s = sum(np.sign(x[j:] - x[i]).sum() for i in range(n-1) for j in [i+1])
    _, counts = np.unique(x, return_counts=True)
    tie = sum(c*(c-1)*(2*c+5) for c in counts if c > 1)
    var_s = (n*(n-1)*(2*n+5)-tie)/18
    if var_s <= 0: z, p = 0.0, 1.0
    elif s > 0:
        z = (s-1)/math.sqrt(var_s); p = 2*(1-norm.cdf(abs(z)))
    elif s < 0:
        z = (s+1)/math.sqrt(var_s); p = 2*(1-norm.cdf(abs(z)))
    else: z, p = 0.0, 1.0
    trend = ("increasing" if s > 0 else "decreasing" if s < 0 else "no trend") if p < alpha else "no significant trend"
    return n, int(s), float(z), float(p), trend

def main():
    if not Path(FILE).exists():
        raise FileNotFoundError(f"Cannot find {FILE!r}. Edit FILE to the full path of Water Quality.csv.")
    os.makedirs(OUT, exist_ok=True)
    df = pd.read_csv(FILE, low_memory=False)
    req = {"ActivityStartDate", "CharacteristicName", "ResultValue"}
    if req - set(df.columns): raise ValueError(f"Missing columns: {req-set(df.columns)}")
    df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
    df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
    df = df.dropna(subset=["ActivityStartDate", "CharacteristicName", "ResultValue"])
    df["Month"] = df["ActivityStartDate"].dt.to_period("M").dt.to_timestamp()
    groups = ["CharacteristicName"]
    if "MonitoringLocationID" in df: groups.insert(0, "MonitoringLocationID")
    rows, plots = [], 0
    for keys, g in df.groupby(groups, dropna=False):
        monthly = g.groupby("Month")["ResultValue"].median().sort_index().dropna()
        if len(monthly) < MIN_MONTHS: continue
        n, s, z, p, trend = mann_kendall(monthly.values, ALPHA)
        slope, intercept, lo, hi = theilslopes(monthly.values, np.arange(len(monthly)), alpha=.95)
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(groups, keys))
        row.update(n_months=n, S=s, z=z, p_value=p, trend=trend,
                   sen_slope_per_month_step=slope, sen_slope_95ci_low=lo,
                   sen_slope_95ci_high=hi, start_month=str(monthly.index.min().date()),
                   end_month=str(monthly.index.max().date()))
        rows.append(row)
        if plots < 25:
            fig, ax = plt.subplots(figsize=(9,4))
            ax.plot(monthly.index, monthly.values, marker="o", label="Monthly median")
            ax.plot(monthly.index, intercept+slope*np.arange(len(monthly)), "--", label=f"Sen slope {slope:.4g}")
            ax.set_title(" / ".join(map(str, keys))); ax.set_xlabel("Month"); ax.set_ylabel("ResultValue (original units)")
            ax.legend(); fig.tight_layout()
            fig.savefig(Path(OUT)/f"trend_{plots+1:02d}.png", dpi=140); plt.close(fig); plots += 1
    result = pd.DataFrame(rows)
    if result.empty:
        print(f"No group had at least {MIN_MONTHS} monthly observations. Adjust MIN_MONTHS if appropriate.")
    else:
        result.sort_values("p_value").to_csv(Path(OUT)/"mann_kendall_sen_slope_results.csv", index=False)
        print(result.sort_values("p_value").head(20).to_string(index=False))
        print(f"Saved results and plots to {OUT}. p < {ALPHA} indicates a significant monotonic trend.")
        print("Slope is per monthly observation step. Classic MK does not adjust for autocorrelation/seasonality.")
if __name__ == "__main__": main()


MonitoringLocationID          CharacteristicName  n_months      S          z  p_value      trend  sen_slope_per_month_step  sen_slope_95ci_low  sen_slope_95ci_high start_month  end_month
          NS01ED0110 Total Nitrogen, mixed forms       266  15944  10.995791      0.0 increasing                  0.001203            0.000965             0.001467  2000-01-01 2023-11-01
          NS01EE0014                      Copper       240 -12577 -10.127630      0.0 decreasing                 -0.001786           -0.002143            -0.001437  2001-01-01 2023-11-01
          NS01DD0016                        Iron       152   6410  10.210999      0.0 increasing                  2.632225            0.043750             4.171102  2006-06-01 2023-10-01
          NS01ED0111 Total Nitrogen, mixed forms       261  14127  10.023669      0.0 increasing                  0.000833            0.000679             0.001000  2000-01-01 2023-11-01
          NS01EE0014                    Chromium       204  -8370